# SMRD - grouped 5-fold cross-validation of the three arms (Colab)

The seed-42 split has test sets of 164 / 298 / 1,285 tweets, too small to tell the arms
apart. Here every tweet is tested once:

* 5 folds, stratified by label; near-duplicate tweets always share a fold.
* Fold *k* = test, fold *k+1* = validation (model selection only), the rest = train.
* Arms unchanged from stages 1-2: **text** (RoBERTa), **graph** (BiGCN), **fusion**.
* One run per fold (seed 42 + fold). Arms are compared tweet by tweet on the pooled predictions.

Checkpoints go to Drive after every epoch; re-running a cell resumes and skips finished folds.
**Run the last cell before switching or disconnecting the runtime.**

In [ ]:
#@title 1. GPU check and dependencies
import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('Runtime -> Change runtime type -> GPU (T4 or L4)')
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')
%pip install -q 'transformers==5.22.1' 'tokenizers==0.23.3' 'safetensors==0.8.0'     'huggingface_hub==2.4.0' 'pandas==3.1.0' 'scikit-learn==1.9.1' 'pytest==9.2.0'

In [ ]:
#@title 2. Mount Drive, get the cv-eval branch
import os, pathlib
from google.colab import drive
drive.mount('/content/drive')

BRANCH = 'cv-eval'
CKPT_ROOT = '/content/drive/MyDrive/smrd/checkpoints/cv'
os.makedirs(CKPT_ROOT, exist_ok=True)

PROJECT_ROOT = pathlib.Path('/content/smrd-reproduction')
if not PROJECT_ROOT.exists():
    !git clone -b {BRANCH} https://github.com/Miyonk/smrd-reproduction.git {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
os.chdir(PROJECT_ROOT)
!git log --oneline -1

In [ ]:
#@title 3. Unit tests (CPU, tiny models, ~2 min) - stop here if anything fails
!python -m pytest tests/test_cv.py tests/test_propagation.py tests/test_encoder_pipeline.py -q

## twitter16

Smallest dataset; a few minutes per arm.

In [ ]:
#@title 4a. twitter16: BiGCN, text, fusion - 5 folds each
for arm in ('graph', 'text', 'fusion'):
    !python -m src.run_cv train --dataset twitter16 --arm {arm} --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 4b. twitter16: pooled comparison
!python -m src.run_cv compare --dataset twitter16 --ckpt-root {CKPT_ROOT}

## twitter15

About twice Twitter16.

In [ ]:
#@title 5a. twitter15: BiGCN, text, fusion - 5 folds each
for arm in ('graph', 'text', 'fusion'):
    !python -m src.run_cv train --dataset twitter15 --arm {arm} --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 5b. twitter15: pooled comparison
!python -m src.run_cv compare --dataset twitter15 --ckpt-root {CKPT_ROOT}

## pheme

Largest; text and fusion take the longest. Node vocabulary is re-selected from each fold's training threads.

In [ ]:
#@title 6a. pheme: BiGCN, text, fusion - 5 folds each
for arm in ('graph', 'text', 'fusion'):
    !python -m src.run_cv train --dataset pheme --arm {arm} --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 6b. pheme: pooled comparison
!python -m src.run_cv compare --dataset pheme --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 7. All comparisons again (saved to Drive)
import glob, shutil
!python -m src.run_cv compare --all-datasets --ckpt-root {CKPT_ROOT}
for p in glob.glob('results/cv/*.json'):
    shutil.copy(p, CKPT_ROOT)

In [ ]:
#@title 8. Flush Drive (run before switching/disconnecting the runtime)
drive.flush_and_unmount()
print('Drive flushed and unmounted')